In [24]:
from typing import Tuple, Union
import numpy as np
import cv2
import os

In [27]:
data_location = "../../data/freiburg_small/"

In [38]:
def getIntrinsics(id: int) -> Union[Tuple[float, float, float, float], None]:
    with open(os.path.join(data_location, "intrinsics.txt")) as file:
        while line := file.readline():
            if line.split(" ")[0] == f"{id:05d}":
                return [float(ele) for ele in line.strip().split(" ")[1:]]
    return None

getIntrinsics(1)

[525.0, 525.0, 319.5, 239.5]

In [39]:
def getDepthMap(id: int) -> Union[np.ndarray, None]:
    def getDepthMapPath(id: int) -> Union[None, str]:
        with open(os.path.join(data_location, "depth.txt")) as file:
            while line := file.readline():
                if line.split(" ")[0] == f"{id:05d}":
                    return line.strip().split(" ")[1]
        return None
    path = getDepthMapPath(id)

    if path is None:
        return None
    
    depth_map = cv2.imread(os.path.join(data_location, path), cv2.IMREAD_UNCHANGED)
    
    height, width = depth_map.shape
    x = np.linspace(0, width - 1, width)
    y = np.linspace(0, height - 1, height)
    xv, yv = np.meshgrid(x, y)

    fX, fY, cX, cY = getIntrinsics(id)

    z = depth_map
    x = (xv - cX) * z / fX
    y = (yv - cY) * z / fY
    points_3d = np.stack((x, y, z), axis=-1)

    return points_3d.reshape(-1, 3)

getDepthMap(1)

array([[-22201.90285714, -16642.74095238,  36482.        ],
       [-23051.51333333, -17333.86952381,  37997.        ],
       [-23017.84285714, -17363.06571429,  38061.        ],
       ...,
       [  7761.51428571,   5854.74857143,  12834.        ],
       [  8008.60666667,   6022.17047619,  13201.        ],
       [  9114.57428571,   6832.3647619 ,  14977.        ]])